# Экспертная валидация извлечения экспериментальных данных из статей (Google Colab)

[![Открыть в Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/top-papers/scientific-extraction-harness/blob/main/notebooks/scientific_experiment_review.ipynb)

Ноутбук заменяет интерактивную работу эксперта с OpenCode-агентом по той же логике, что
[ноутбук legal-agent-harness](https://colab.research.google.com/github/lawful-good-project/legal-agent-harness/blob/main/notebooks/law_extract_fixed.ipynb):
**загрузить статью → Run all → модель извлекает данные → оценить результат в форме → отправить в Hugging Face**.

1. Эксперт нажимает **«Загрузить статью»** (файл или ссылка); ноутбук ждёт загрузку и продолжает работу.
2. Текст статьи разбирается офлайн и передаётся **бесплатным LLM-провайдерам `g4f`** (без ключа). Модель возвращает
   кандидаты `EvidenceRecord` по контракту [`docs/data-contract.md`](https://github.com/top-papers/scientific-extraction-harness/blob/main/docs/data-contract.md):
   `page`, `modality`, `locator`, `field`, `value`, `unit`, `confidence`, `evidence_quote`.
3. Каждый кандидат сверяется с текстом: цитата должна встречаться дословно, страница определяется по цитате,
   запись без свидетельства отбрасывается ещё до формы.
4. Создаётся **автономная офлайн HTML-форма** (`expert_review_*.html`): по каждому значению «верно / исправить / нет в
   источнике» (и отдельно «null» для значения), статус `approved`/`rejected`, пометка, оценка 0–10, комментарий, публикация.
5. Только одобренные экспертом записи уходят в датасет Hugging Face: YAML `reviewed-dataset.v1` (кандидаты, provenance
   проверки и явное событие `expert_validation`) через Commit API с `create_pr=1`.

> Это проверка качества извлечения, а не научное заключение. Ни одна запись не получает статус `approved` автоматически —
> только от эксперта. Если свидетельства нет или оно неоднозначно, эксперт ставит `rejected` и пишет заметку.

**Перед запуском** откройте слева **🔑 Секреты (Secrets)** и добавьте:
- `HF_TOKEN` — токен Hugging Face с правом записи в целевой датасет;
- `top_papers_token` — только если статьи берутся из приватного GitHub (Bearer-заголовок; это не токен HF).

Токены не печатаются и не попадают ни в HTML-форму, ни в публикуемую запись. Исходная статья (PDF/книга) в датасет не отправляется.

## 1. Установка зависимостей

`pypdf` — текстовый слой PDF; `g4f` — бесплатные публичные провайдеры LLM. Платных SDK нет: извлечение не требует токена HF.

In [ ]:
%pip install -q -U pypdf g4f
print("Dependencies ready: pypdf, g4f")

## 2. Конфигурация и каскад проверки моделей g4f

Модели проверяются каскадом из трёх ступеней (дизайн: `docs/design-vision-chart-validation.md`, раздел 4.1):
1. **Работоспособность** — текстовый ping «Reply exactly: OK» по каталогу g4f; 402/таймауты/отказные фразы
   («Sign up…») отсекаются сразу и не тратят время на дорогие проверки.
2. **Мультимодальность** — image-ping на программной PNG-диаграмме (3 столбика): модель обязана назвать «3»,
   иначе она не читает изображения. Проверяются только живые модели, помеченные `VisionModel`, плюс
   `VISION_MODELS_EXTRA`; поиск останавливается на первой прошедшей.
3. **Выбор режима** — `vision` (есть мультимодальная модель: графики будут читаться ею), `text` (живые есть,
   мультимодальных нет — работа как раньше), `fallback` (не жива ни одна — Pollinations-резерв, только текст).

Аудит каждой модели (`MODEL_AUDIT`: вердикты ступеней и причины) печатается и попадает в форму, поэтому видно,
почему выбран именно этот режим. Включение vision-контура задаёт `ENABLE_VISION`. OCR-фолбэк (Tesseract) для
текстового режима планируется отдельным PR (см. дизайн, раздел 4.1.3).

Чаще всего правят: `HF_REPO_ID` (целевой датасет), `URLS` / `LOCAL_PATH` (источник статьи), `STRICT_SCHEMA`, `CHUNK_PAGES`.

In [ ]:
import asyncio, json, datetime as dt, io, os, re, time

MODE = "single"                  # "single" или "batch"
URLS = ""                        # batch: ссылки через запятую; пусто — файл кнопкой
LOCAL_PATH = ""                  # напр. "/content/drive/MyDrive/paper.pdf"; пусто — кнопка загрузки
SKIP_UPLOAD = False              # True — не спрашивать, взять файлы статьи из /content

HF_REPO_ID = "chaotic-good-project/scientific-extraction-review"  # DEFAULT_REPO_ID из src/scientific_extraction_harness/hf_dataset.py
HF_TOKEN_SECRET_NAME = "HF_TOKEN"
TOP_PAPERS_TOKEN_SECRET_NAME = "top_papers_token"   # только для приватных GitHub-источников

MODEL = ""                       # заполняется каскадом ниже (режим/модель)
G4F_CONCURRENCY = 3              # ограничение параллельных запросов
MAX_ATTEMPTS = 4
MIN_INTERVAL_SEC = 2             # пауза между запросами
TIMEOUT_SEC = 180
MAX_TEXT_CHARS = 24000           # сколько символов видит модель за один запрос
CHUNK_PAGES = 6                  # страниц в одном фрагменте
MAX_UPLOAD_MB = 25

STRICT_SCHEMA = True             # строгая схема экспериментальных полей
MAX_EXPERIMENTS = 12             # верхняя граница кандидатов на статью
MAX_RECORDS_PER_EXPERIMENT = 30
ALLOW_RULES_FALLBACK = False     # True — черновик по регулярным правилам, если g4f недоступен
REMOTE_FORM_UPLOAD = True        # разрешить отправку прямо из HTML-формы в браузере

EXPERT_NAME = ""                 # необязательно: подпись эксперта
LICENSE_DECLARED = "unknown"     # что эксперт знает о правах на статью, напр. "CC-BY-4.0"

# --- каскад проверки моделей (дизайн docs/design-vision-chart-validation.md, 4.1) ---
ENABLE_VISION = bool(ENABLE_VISION) if "ENABLE_VISION" in dir() else True  # ступень 2: искать мультимодальную модель
VISION_MODELS_EXTRA = list(VISION_MODELS_EXTRA) if "VISION_MODELS_EXTRA" in dir() else []  # доп. имена для image-ping, напр. ["qwen-2.5-vl-72b"]
VISION_FALLBACK_TEXT = True      # текстовый контур остаётся всегда, vision только добавляет графики
PROBE_TIMEOUT_SEC = 45           # таймаут ping-а, меньше TIMEOUT_SEC
PROBE_MAX_ALIVE = 6              # ступень 1 останавливается после стольких живых (не весь каталог пингуем в loop)

MODEL_MODE = "text"              # итог каскада: "vision" | "text" | "fallback"
ACTIVE_G4F_MODEL = None          # выбранная модель (None при fallback)
G4F_MODELS_TO_PROBE = []         # живые текстовые модели каскада (для перебора при извлечении)
MODEL_AUDIT = []                 # [{model, alive, alive_why, vision, vision_why}] — в CFG и issues формы


def _refusal_like(text):
    """«Sign up и repeat…», «I can't», пусто или слишком длинно — не содержательный ответ."""
    t = (text or "").strip().lower()
    return (not t or len(t) > 400 or "sign up" in t or "i can't" in t
            or "cannot view" in t or "log in" in t or "subscribe" in t)


def _probe_client():
    from g4f.client import Client
    return Client(timeout=PROBE_TIMEOUT_SEC)


def _completion_text(r):
    return (r.choices[0].message.content or "").strip() if r and r.choices else ""


def _catalog_names():
    """Ступень 0: все имена моделей g4f (+ ручные дополнения)."""
    try:
        from g4f import models as g4f_models
        reg = g4f_models.ModelRegistry.all_models()
        names = [n for n, m in reg.items() if n]
    except Exception:
        names = []
    if not names:  # старые версии g4f без ModelRegistry
        try:
            from g4f.models import ModelUtils
            names = list((getattr(ModelUtils, "convert", None) or {}).keys())
        except Exception:
            names = []
    if not names:
        names = ["gpt-4o", "llama-3.2-11b"]
    for extra in VISION_MODELS_EXTRA:
        if extra not in names:
            names.append(extra)
    return list(dict.fromkeys(names))


def _vision_marked_names():
    """Имена, помеченные VisionModel в каталоге g4f (метка ≠ гарантия, проверит image-ping)."""
    try:
        from g4f import models as g4f_models
        reg = g4f_models.ModelRegistry.all_models()
        return {n for n, m in reg.items() if isinstance(m, g4f_models.VisionModel) and n}
    except Exception:
        return set()


def probe_alive(model):
    """Ступень 1: модель жива — текстовый ping отвечает «OK» без отказных фраз."""
    try:
        r = _probe_client().chat.completions.create(
            model=model, temperature=0,
            messages=[{"role": "user", "content": "Reply exactly: OK"}])
        text = _completion_text(r)
        return ("ok" in text.lower()) and not _refusal_like(text), text
    except Exception as exc:
        return False, f"{type(exc).__name__}: {str(exc)[:80]}"


def probe_vision(model):
    """Ступень 2: модель читает изображения — image-ping на PNG-диаграмме с 3 столбиками.

    ok только если ответ содержательный и называет «3»: отсекает и отказные фразы
    («Sign up…»), и модели, которые не смотрят на картинку.
    """
    try:
        r = _probe_client().chat.completions.create(
            model=model, temperature=0,
            messages=[{"role": "user", "content":
                       "This image shows a bar chart. How many bars are there? "
                       "Answer with a single number only."}],
            image=VISION_PROBE_IMAGE, image_name="probe_bars.png")
        text = _completion_text(r)
        if _refusal_like(text):
            return False, f"refusal-like: {text[:60]!r}"
        if not re.search(r"\b3\b", text):
            return False, f"wrong answer: {text[:60]!r}"
        return True, text[:40]
    except Exception as exc:
        return False, f"{type(exc).__name__}: {str(exc)[:80]}"


def make_probe_image():
    """PNG 128×96: белое поле, чёрные оси, 3 столбика разной высоты. Pillow, без сети."""
    from PIL import Image, ImageDraw
    img = Image.new("RGB", (128, 96), "white")
    d = ImageDraw.Draw(img)
    d.rectangle([12, 8, 15, 88], fill="black")    # ось Y
    d.rectangle([12, 85, 118, 88], fill="black")  # ось X
    for k, h in enumerate((30, 55, 78)):          # 3 бара разной высоты
        x = 24 + k * 36
        d.rectangle([x, 88 - h, x + 22, 86], fill="black")
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


VISION_PROBE_IMAGE = make_probe_image()


def select_models(verbose=True):
    """Каскад: каталог → ступень 1 (живость) → ступень 2 (мультимодальность) → режим.

    Возвращает (mode, active_model, vision_models, audit).
    Ступень 1 не пингует весь каталог бесконечно: после PROBE_MAX_ALIVE живых
    или исчерпания списка — стоп; image-ping только для живых VisionModel/EXTRA,
    остановка на первой прошедшей.
    """
    audit, alive_order = [], []
    catalog = _catalog_names()
    alive_count = 0
    if verbose:
        print(f"Каскад проверки моделей g4f: кандидатов {len(catalog)}")
    for m in catalog:
        if alive_count >= PROBE_MAX_ALIVE and m not in VISION_MODELS_EXTRA:
            audit.append({"model": m, "alive": None, "alive_why": "не проверялась (лимит живых достигнут)",
                          "vision": None, "vision_why": ""})
            continue
        ok, why = probe_alive(m)
        if not ok and not why:
            why = "empty reply"
        audit.append({"model": m, "alive": ok, "alive_why": why if not ok else "",
                      "vision": None, "vision_why": ""})
        if verbose:
            print(f"  [1] {m}: {'жива' if ok else 'недоступна — ' + why}")
        if ok:
            alive_order.append(m)
            alive_count += 1

    vision = []
    if ENABLE_VISION:
        marked = _vision_marked_names()
        candidates = [m for m in alive_order if m in marked or m in VISION_MODELS_EXTRA]
        if not candidates and verbose:
            print("  [2] нет живых кандидатов для image-ping (VisionModel/EXTRA)")
        for m in candidates:
            ok, why = probe_vision(m)
            if ok:
                vision.append(m)
            for row in audit:
                if row["model"] == m:
                    row["vision"], row["vision_why"] = ok, why if not ok else ""
            if verbose:
                print(f"  [2] {m}: {'multimodal' if ok else 'не прошла image-ping — ' + why}")
            if vision:
                break  # достаточно одной рабочей vision-модели
    elif verbose:
        print("  [2] vision-контур отключён (ENABLE_VISION = False)")

    if vision:
        mode, active = "vision", vision[0]
        if verbose:
            print(f"✅ Режим vision: {active} (текст остаётся основной, графики добавятся vision-контуром)")
    elif alive_order:
        mode, active = "text", alive_order[0]
        if verbose:
            print(f"⚠️ Мультимодальных моделей не найдено: режим text ({active}). "
                  "Значения только с картинок будут помечены null (правило промпта).")
    else:
        mode, active = "fallback", None
        if verbose:
            print("⚠️ Ни одна модель g4f не прошла проверку: режим fallback — "
                  "перебор резервных источников Pollinations (только текст).")
    return mode, active, vision, audit


MODEL_MODE, ACTIVE_G4F_MODEL, _vision_models, MODEL_AUDIT = select_models()
G4F_MODELS_TO_PROBE = [row["model"] for row in MODEL_AUDIT if row["alive"]] or _catalog_names()
MODEL = ACTIVE_G4F_MODEL or ("pollinations-openai" if MODEL_MODE == "fallback" else "")

CFG = dict(mode=MODE, urls=URLS, repo=HF_REPO_ID, model=MODEL, model_mode=MODEL_MODE, strict=STRICT_SCHEMA,
           max_attempts=MAX_ATTEMPTS, min_interval=MIN_INTERVAL_SEC, timeout=TIMEOUT_SEC,
           max_chars=MAX_TEXT_CHARS, chunk_pages=CHUNK_PAGES, max_mb=MAX_UPLOAD_MB,
           expert=EXPERT_NAME, license=LICENSE_DECLARED,
           created_at=dt.date.today().isoformat(),
           run_id="run-" + dt.datetime.now().strftime("%Y%m%d%H%M%S"))
print("активная модель:", MODEL or "(нет)", "| режим:", MODEL_MODE)
print(json.dumps(CFG, ensure_ascii=False, indent=2))

## 3. Загрузка статьи

Появится кнопка **«Загрузить статью»**: выберите PDF / DOCX / TXT / MD / HTML / RTF. Ноутбук ждёт загрузку (до 15 минут)
и продолжает выполнение следующих ячеек автоматически — повторять Run all не нужно.

Если файл уже лежит в Colab или на Google Drive, укажите `LOCAL_PATH` выше (Drive сначала подключите: 📁 → Mount Drive).
Ссылки на статьи указываются в `URLS` — до 5 через запятую. Для URL на `github.com` из приватного репозитория
используется секрет `top_papers_token` только как заголовок `Authorization`; значение не печатается.

Сканы без текстового слоя не обрабатываются: ноутбук предупредит об этом и не будет придумывать данные.

In [ ]:
import glob, hashlib, html, io, urllib.parse, urllib.request, zipfile
from pathlib import Path

ALLOWED_EXT = (".pdf", ".docx", ".txt", ".md", ".html", ".htm", ".rtf")
UPLOAD_DIR = Path("/content/uploads" if os.path.isdir("/content") else "uploads")

def _secret(name):
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
    except Exception:
        value = os.environ.get(name)
    return (value or "").strip()

TOP_PAPERS_TOKEN = _secret(TOP_PAPERS_TOKEN_SECRET_NAME)

def _safe_name(name):
    base = os.path.basename(str(name or "").replace("\\", "/")).strip()
    base = re.sub(r"[^\w.\- ()\u0400-\u04FF]+", "_", base)[:140]
    ext = os.path.splitext(base)[1].lower()
    if not base or ext not in ALLOWED_EXT:
        raise ValueError(f"формат «{ext or base}» не поддерживается; допустимо: {', '.join(ALLOWED_EXT)}")
    return base

def _store(name, raw):
    if not raw:
        raise ValueError("пустой файл")
    if len(raw) > CFG["max_mb"] * 1024 * 1024:
        raise ValueError(f"файл больше {CFG['max_mb']} МБ")
    UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
    path = UPLOAD_DIR / _safe_name(name)
    path.write_bytes(raw)
    return str(path)

def _fetch_url(url):
    url = url.strip()
    if not url.lower().startswith(("http://", "https://")):
        raise ValueError("ссылка должна начинаться с http:// или https://")
    headers = {"User-Agent": "scientific-extraction-colab/1.0"}
    if "github.com" in urllib.parse.urlparse(url).netloc and TOP_PAPERS_TOKEN:
        headers["Authorization"] = "Bearer " + TOP_PAPERS_TOKEN
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=120) as resp:
        raw = resp.read(CFG["max_mb"] * 1024 * 1024 + 1)
    name = os.path.basename(urllib.parse.unquote(urllib.parse.urlparse(url).path)) or "article.pdf"
    if os.path.splitext(name)[1].lower() not in ALLOWED_EXT:
        name = (os.path.splitext(name)[0] or "article") + ".pdf"
    return _store(name, raw)

files = []
url_list = [u.strip() for u in str(CFG["urls"] or "").split(",") if u.strip()]
if url_list:
    if len(url_list) > 5:
        print("⚠️ Обрабатываются только первые 5 ссылок")
    for url in url_list[:5]:
        try:
            files.append(_fetch_url(url))
            print("↓", os.path.basename(files[-1]))
        except Exception as exc:
            print("✗", url, "—", type(exc).__name__, str(exc)[:160])
elif LOCAL_PATH:
    files = [LOCAL_PATH]
elif SKIP_UPLOAD:
    files = sorted(p for ext in ALLOWED_EXT for p in glob.glob(f"/content/*{ext}"))
else:
    try:
        from google.colab import files as colab_files  # type: ignore
        print("Выберите статью (PDF/TXT/HTML/DOCX/RTF)…")
        uploaded = colab_files.upload()
        for name, data in uploaded.items():
            try:
                files.append(_store(name, data))
                print(f"Uploaded {name} ({len(data)/1048576:.2f} MiB)")
            except Exception as exc:
                print(f"Skipped {name}: {type(exc).__name__}: {exc}")
    except Exception as exc:
        print("Colab upload failed:", type(exc).__name__, str(exc)[:250])
        print("Загрузите файл через панель 📁 и поставьте SKIP_UPLOAD = True.")
files = [f for f in dict.fromkeys(files) if os.path.isfile(f)]
print("\nДокументов к обработке:", len(files))
for f in files:
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1048576:.2f} MiB)")

## 4. Разбор статьи в текст с номерами страниц

Офлайн-парсинг без OCR: `pypdf` для PDF (страницы помечаются `[[PAGE n]]`, чтобы модель могла дать provenance),
`zipfile` для DOCX, регулярные правила для HTML/RTF/TXT. Сохраняются SHA-256, число страниц/символов и смещения
маркеров страниц; сам текст статьи в публикуемую запись не попадает.

In [ ]:
def _rtf_like(data: bytes) -> bool:
    head = data.lstrip()[:5].lower().startswith(b"{\\rtf")
    return head and bool(re.search(rb"\\'[0-9a-fA-F]{2}|\\u-?\d+", data[:20000]))

def text_from_bytes(name: str, data: bytes) -> str:
    ext = Path(name).suffix.lower()
    if ext in (".txt", ".md", ".csv", ".xml"):
        for enc in ("utf-8", "cp1251", "koi8-r"):
            try:
                return data.decode(enc)
            except UnicodeDecodeError:
                continue
        return data.decode("utf-8", "replace")
    if ext in (".html", ".htm"):
        s = data.decode("utf-8", "replace")
        s = re.sub(r"(?is)<(script|style)[^>]*>.*?</\1>", " ", s)
        return html.unescape(re.sub(r"<[^>]+>", " ", s))
    if ext == ".rtf":
        s = data.decode("latin-1", "replace")
        if _rtf_like(data):
            s = re.sub(r"\\u(-?\d+)\??", lambda m: chr(int(m.group(1)) % 65536), s)
            s = re.sub(r"\\'([0-9a-fA-F]{2})", lambda m: bytes([int(m.group(1), 16)]).decode("cp1251", "replace"), s)
            s = re.sub(r"\\(?:par[d]?|line|tab)\b ?", "\n", s)
            s = re.sub(r"\\[a-z]+-?\d* ?", " ", s)
            return s.replace("{", " ").replace("}", " ")
        for enc in ("utf-8", "cp1251"):
            try:
                return re.sub(r"^\s*\{\\rtf1[^ ]* ?|\\par?}\s*$", "", data.decode(enc)).replace("\\par", "\n")
            except UnicodeDecodeError:
                continue
        return data.decode("utf-8", "replace")
    if ext == ".docx":
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            xml = z.read("word/document.xml").decode("utf-8", "replace")
        xml = xml.replace("</w:p>", "\n")
        return html.unescape(re.sub(r"<[^>]+>", "", xml))
    if ext == ".pdf":
        try:
            from pypdf import PdfReader
            reader = PdfReader(io.BytesIO(data))
            pages = [f"[[PAGE {i}]]\n" + (page.extract_text() or "") for i, page in enumerate(reader.pages[:300], 1)]
            text = "\n".join(pages)
            if text.strip():
                return text
        except Exception as exc:
            print("pypdf не смог прочитать PDF:", type(exc).__name__)
        s = data.decode("latin-1", "ignore")
        return "[[PAGE 1]]\n" + "\n".join(re.findall(r"[\x20-\x7e\u0400-\u04FF\u2116\u00ab\u00bb\u2013\u2014]{8,}", s))
    raise ValueError(f"Неподдерживаемый формат: {ext}")

PAGE_RE = re.compile(r"\[\[PAGE\s+(\d+)\]\]")

def normalize(text: str) -> str:
    text = text.replace("\xa0", " ").replace("\u00ad", "")
    lines = [re.sub(r"[ \t]+", " ", ln).strip() for ln in text.splitlines()]
    out, blank = [], 0
    for ln in lines:
        if not ln:
            blank += 1
            if blank > 1:
                continue
        else:
            blank = 0
        out.append(ln)
    return "\n".join(out).strip()

def page_offsets(text: str):
    return [(m.start(), int(m.group(1))) for m in PAGE_RE.finditer(text)]

def page_at(offsets, pos: int) -> int:
    page = 1
    for start, number in offsets:
        if start <= pos:
            page = number
        else:
            break
    return page

def load_documents():
    docs = []
    for path in files:
        try:
            data = Path(path).read_bytes()
            text = normalize(text_from_bytes(path, data))
        except Exception as exc:
            print(f"✗ {path}: {type(exc).__name__}: {exc}")
            continue
        offsets = page_offsets(text)
        docs.append({"filename": os.path.basename(path), "size_bytes": len(data), "text": text,
                     "sha256": hashlib.sha256(data).hexdigest(), "pages": len(offsets) or 1,
                     "offsets": offsets,
                     "document_id": "doc-" + hashlib.sha256(data).hexdigest()[:16]})
    return docs

DOCS = load_documents()
for d in DOCS:
    print(f"✓ {d['filename']}: {len(d['text']):,} символов, страниц: {d['pages']}, sha256={d['sha256'][:12]}…")
    body = PAGE_RE.sub("", d["text"]).strip()
    if len(body) < 300 or re.search(r"/(Type|Filter|Font|obj)\b", body):
        print("  ⚠️ Текстовый слой почти пустой — вероятно, скан без OCR. Предоставьте текстовый PDF/DOCX/TXT:"
              " иначе извлечение будет вымыслом.")
if not DOCS:
    print("Нет документов для обработки (см. ячейку 3).")

## 5. Схема извлечения экспериментальных данных

Схема описывает дизайн эксперимента и значения его исходов. Термины следуют контракту `EvidenceRecord`, поэтому каждый
измеренный показатель — отдельная запись со своими страницей, modality, locator, единицей и цитатой.

**Статусы:** `candidate` — кандидат модели; `approved` — эксперт подтвердил по источнику; `rejected` — лишнее или неверное.
`value: null` означает «в статье значение не заявлено / неоднозначно»: эксперт подтверждает null или вписывает значение.
Значение не появляется без дословной цитаты: запись без найденной цитаты отбрасывается до формы.

In [ ]:
PROFILES = ["experiment", "observational"]
MODALITIES = ["text", "table", "figure", "scan"]
STATUSES = ["candidate", "approved", "rejected"]
VALUE_TYPES = ["number", "rate", "category", "count", "statistic", "qualitative"]
EXPERIMENT_FIELDS = ["experiment_id", "profile", "title", "research_question", "hypothesis", "design",
                     "sample_size", "population", "methods", "intervention", "control", "conclusion"]
RESULT_FIELDS = ["record_id", "document_id", "page", "modality", "locator", "field", "value", "unit",
                 "value_type", "confidence", "status", "evidence_quote", "reviewer_note"]
FIELD_ALIASES = {
    "metric": "field", "variable": "field", "measure": "field", "outcome": "field",
    "result": "value", "measured_value": "value",
    "units": "unit", "uom": "unit",
    "page_number": "page",
    "source_modality": "modality", "origin": "modality",
    "coordinate": "locator", "location": "locator",
    "quote": "evidence_quote", "snippet": "evidence_quote", "evidence": "evidence_quote",
    "certainty": "confidence", "confidence_score": "confidence",
    "note": "reviewer_note", "comment": "reviewer_note",
}

SCHEMA_SPEC = """{
  "experiments": [
    {
      "experiment_id": "exp-1",
      "profile": "experiment|observational",
      "title": "short title of the experiment",
      "research_question": "question the experiment answers",
      "hypothesis": "hypothesis or null",
      "design": "randomized controlled trial | field trial | cohort | case study | ...",
      "sample_size": "as reported, e.g. 'n=120' or null",
      "population": "who/what was studied",
      "methods": "measurement or analysis procedure",
      "intervention": "treatment / treatment levels",
      "control": "control or comparison group or null",
      "conclusion": "authors' conclusion for this experiment",
      "results": [
        {
          "page": 3,
          "modality": "text|table|figure|scan",
          "locator": "p.3 para 2 | Table 2 row 'yield', column '200 kg N' | Figure 1 panel b",
          "field": "normalized variable name, e.g. 'grain yield'",
          "value": "verbatim value as printed, e.g. '7.2', or null if not reported/ambiguous",
          "unit": "verbatim unit, e.g. 't/ha', or null",
          "value_type": "number|rate|count|statistic|category|qualitative",
          "confidence": 0.0,
          "status": "candidate",
          "evidence_quote": "verbatim sentence or table row that supports the value",
          "reviewer_note": ""
        }
      ]
    }
  ]
}"""

SYSTEM_PROMPT = (
    "You extract experimental data from a scientific article into strict JSON. Answer ONLY with a JSON object, no markdown.\n"
    "Schema: " + SCHEMA_SPEC + "\n"
    "Rules:\n"
    "1. Only report values supported by the given text. Copy values and units exactly as printed; never convert or recompute.\n"
    "2. If a number is ambiguous, missing, or only readable from an image you cannot see, set value to null, confidence <= 0.4 and explain in reviewer_note.\n"
    "3. evidence_quote must be a verbatim substring of the provided fragment (short sentence or table row). Never paraphrase inside it.\n"
    "4. page is the [[PAGE n]] marker the evidence appears under; if the fragment has no marker, use page 1.\n"
    "5. locator names the exact place: table number and row/column, figure number and panel, or 'para k'.\n"
    "6. Include experimental conditions (dose, duration, temperature, site), sample sizes, uncertainty (SD/SE/CI/p) and comparison groups when stated.\n"
    "7. status is always 'candidate'. Field names use lower-case snake_case English nouns.\n"
    "8. Do not merge several numbers into one record; one record = one measured value.\n"
    "9. Text may be truncated: '…[middle of article skipped]…' means part of the article is not shown; do not invent what is missing.\n"
    "10. If the fragment contains no experimental data, return {\"experiments\": []}."
)

print("строгая схема:", CFG["strict"], "| полей результата:", len(RESULT_FIELDS), "| профили:", PROFILES)

## 6. Извлечение через бесплатные провайдеры g4f

Текст статьи режется на фрагменты по `CHUNK_PAGES` страниц; в каждом фрагменте модель видит маркеры `[[PAGE n]]`.
Ответы объединяются и дедуплицируются. Стратегия та же, что в ноутбуке legal-agent-harness: восстановление обрезанного
JSON, повтор при ошибке, перебор провайдеров (`g4f` → Pollinations `gpt-oss-20b` → `openai`), предупреждения вместо
выдуманных данных.

In [ ]:
def _window_text(text, limit):
    if len(text) <= limit:
        return text
    marker = "\n\n…[middle of article skipped]…\n\n"
    budget = max(limit - len(marker), 1000)
    head = int(budget * 0.6)
    return text[:head] + marker + text[-(budget - head):]

def split_fragments(doc):
    """Фрагменты по маркерам страниц и по символам; возвращают список (текст, страницы)."""
    text, offsets = doc["text"], doc["offsets"]
    if not offsets:
        size = max(2000, CFG["max_chars"])
        return [(_window_text(text[i:i+size], size), [1]) for i in range(0, len(text), size)]
    bounds, i = [], 0
    while i < len(offsets):
        start, first_page = offsets[i][0], offsets[i][1]
        j = i
        while j + 1 < len(offsets) and (offsets[j + 1][1] - first_page) < CFG["chunk_pages"]:
            j += 1
        end = offsets[j + 1][0] if j + 1 < len(offsets) else len(text)
        bounds.append((start, end, [n for _, n in offsets[i:j + 1]]))
        i = j + 1
    pieces = []
    for start, end, pages in bounds:
        piece = text[start:end]
        if len(piece) > CFG["max_chars"]:
            for k in range(0, len(piece), CFG["max_chars"]):
                pieces.append((_window_text(piece[k:k+CFG["max_chars"]], CFG["max_chars"]), pages))
        else:
            pieces.append((piece, pages))
    return pieces

class _Truncated(Exception):
    pass

def _clean_json(text):
    s = (text or "").strip()
    for a, b in (("\u201c", '"'), ("\u201d", '"'), ("\u00ab", '"'), ("\u00bb", '"'), ("\u2018", "'"), ("\u2019", "'")):
        s = s.replace(a, b)
    i = s.find("{")
    if i < 0:
        raise ValueError("в ответе модели нет JSON-объекта")
    s = s[i:].rstrip()
    if s.endswith("```"):
        s = s[: s.rindex("```")].rstrip()
    return s

def _parse_structural(s):
    """JSON с послаблениями: «сырая» кавычка внутри значения и обрыв ответа (_Truncated)."""
    n = len(s)
    pos = [0]

    def ws():
        while pos[0] < n and s[pos[0]] in " \t\r\n":
            pos[0] += 1

    def string(expect_key):
        pos[0] += 1
        buf = []
        while True:
            if pos[0] >= n:
                raise _Truncated
            ch = s[pos[0]]
            if ch == "\\" and pos[0] + 1 < n:
                nxt = s[pos[0] + 1]
                if nxt in '"\\/bfnrt':
                    buf.append({"b": "\b", "f": "\f", "n": "\n", "r": "\r", "t": "\t"}.get(nxt, nxt))
                    pos[0] += 2
                else:
                    buf.append('"')
                    pos[0] += 1
                continue
            if ch == '"':
                j = pos[0] + 1
                while j < n and s[j] in " \t\r\n":
                    j += 1
                nxt = s[j] if j < n else ""
                if nxt in (":", ",", "}", "]") or (expect_key and nxt == ""):
                    pos[0] += 1
                    return "".join(buf)
                if nxt == "":
                    raise _Truncated
                raw_quote = nxt == '"' or bool(re.match(r'"\s*[:,}]', s[j:j + 4])) or expect_key
                buf.append('"' if raw_quote else ch)   # «сырая» кавычка внутри значения остаётся символом
                pos[0] += 1
                continue
            buf.append(ch)
            pos[0] += 1

    def value():
        ws()
        if pos[0] >= n:
            raise _Truncated
        c = s[pos[0]]
        if c == "{":
            return obj()
        if c == "[":
            return arr()
        if c == '"':
            return string(False)
        start = pos[0]
        while pos[0] < n and s[pos[0]] not in ",:]}]\n":
            pos[0] += 1
        word = s[start:pos[0]].strip()
        if word in ("true", "false", "null"):
            return {"true": True, "false": False, "null": None}[word]
        try:
            return int(word)
        except ValueError:
            try:
                return float(word)
            except ValueError:
                return word

    def obj():
        pos[0] += 1
        out = {}
        ws()
        if pos[0] >= n:
            raise _Truncated
        if s[pos[0]] == "}":
            pos[0] += 1
            return out
        while True:
            ws()
            if pos[0] >= n:
                raise _Truncated
            if s[pos[0]] != '"':
                raise ValueError("ожидался ключ")
            key = string(True)
            ws()
            if pos[0] >= n:
                raise _Truncated
            if s[pos[0]] != ":":
                raise ValueError("ожидалось ':'")
            pos[0] += 1
            out[key] = value()
            ws()
            if pos[0] >= n:
                return out
            if s[pos[0]] == ",":
                pos[0] += 1
                ws()
                if pos[0] < n and s[pos[0]] == "}":
                    pos[0] += 1
                    return out
                continue
            if s[pos[0]] == "}":
                pos[0] += 1
                return out
            raise ValueError("ожидалось ',' или '}'")

    def arr():
        pos[0] += 1
        out = []
        ws()
        if pos[0] < n and s[pos[0]] == "]":
            pos[0] += 1
            return out
        while True:
            out.append(value())
            ws()
            if pos[0] >= n:
                return out
            if s[pos[0]] == ",":
                pos[0] += 1
                ws()
                if pos[0] < n and s[pos[0]] == "]":
                    pos[0] += 1
                    return out
                continue
            if s[pos[0]] == "]":
                pos[0] += 1
                return out
            raise ValueError("ожидалось ',' или ']'")

    return value()

def parse_json_recovering(text):
    """(obj, truncated): переживает markdown, «сырые» кавычки и обрыв ответа модели."""
    s = _clean_json(text)
    try:
        return json.loads(s), False
    except json.JSONDecodeError:
        pass
    try:
        return _parse_structural(s), False
    except _Truncated:
        pass
    for cut in range(len(s) - 1, 0, -1):
        if s[cut] not in "]}":
            continue
        try:
            cand = _parse_structural(s[: cut + 1])
        except (_Truncated, ValueError, json.JSONDecodeError):
            continue
        if isinstance(cand, dict) and isinstance(cand.get("experiments"), list):
            return cand, True
    # Обрыв внутри незакрытой строки: обрезаем до неё и закрываем незакрытые скобки.
    quote_positions = [m.start() for m in re.finditer('"', s)]
    if len(quote_positions) % 2 == 1:
        head = s[: quote_positions[-1]].rstrip().rstrip(",")
        stack = []
        in_str = False
        for ch in head:
            if ch == '"':
                in_str = not in_str
            elif not in_str and ch in "[{":
                stack.append(ch)
            elif not in_str and ch in "]}" and stack:
                stack.pop()
        try:
            cand = _parse_structural(head + "".join("}" if ch == "{" else "]" for ch in reversed(stack)))
            if isinstance(cand, dict) and isinstance(cand.get("experiments"), list):
                return cand, True
        except (_Truncated, ValueError, json.JSONDecodeError):
            pass
    raise ValueError("не удалось восстановить JSON из ответа модели")

def _run_coroutine(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()

def _request_sync(backend, model, system, user):
    if backend == "g4f":
        from g4f.client import Client
        r = Client().chat.completions.create(
            model=model, messages=[{"role": "system", "content": system}, {"role": "user", "content": user}], temperature=0)
        return (r.choices[0].message.content or "").strip() if r and r.choices else ""
    payload = {"model": model, "temperature": 0,
               "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
    req = urllib.request.Request(
        "https://text.pollinations.ai/openai", data=json.dumps(payload).encode(), method="POST",
        headers={"Content-Type": "application/json", "User-Agent": "scientific-extraction-colab/1.0"})
    with urllib.request.urlopen(req, timeout=CFG["timeout"]) as resp:
        data = json.loads(resp.read().decode())
    return ((data.get("choices") or [{}])[0].get("message") or {}).get("content", "").strip()

async def _request_async(backend, model, system, user):
    if backend == "g4f":
        try:
            from g4f.client import AsyncClient
            r = await AsyncClient().chat.completions.create(
                model=model, messages=[{"role": "system", "content": system}, {"role": "user", "content": user}], temperature=0)
            return (r.choices[0].message.content or "").strip() if r and r.choices else ""
        except (ImportError, AttributeError, TypeError, NotImplementedError):
            pass
    return await asyncio.to_thread(_request_sync, backend, model, system, user)

def llm_extract(doc):
    """Извлекает эксперименты по фрагментам; возвращает (experiments, truncated, errors)."""
    fragments = split_fragments(doc)
    candidates = []
    if ACTIVE_G4F_MODEL:
        candidates.append(("g4f", ACTIVE_G4F_MODEL))
    candidates.extend(("g4f", m) for m in G4F_MODELS_TO_PROBE if m != ACTIVE_G4F_MODEL)
    candidates.extend([("pollinations", "gpt-oss-20b"), ("pollinations", "openai")])

    async def one_fragment(idx, piece, pages, sem):
        user = (f"Fragment {idx}/{len(fragments)} of the article '{doc['filename']}'. Pages in this fragment: "
                f"{pages[0]}-{pages[-1]}. Return JSON per the schema.\n\nARTICLE FRAGMENT:\n{piece}")
        failures = []
        async with sem:
            for backend, model in candidates:
                try:
                    content = await _request_async(backend, model, SYSTEM_PROMPT, user)
                    if not content:
                        raise ValueError("empty content")
                    result, was_truncated = parse_json_recovering(content)
                    if not isinstance(result, dict) or not isinstance(result.get("experiments"), list):
                        raise ValueError("missing experiments list")
                    print(f"  ✓ фрагмент {idx}/{len(fragments)}: {backend}/{model}, экспериментов={len(result['experiments'])}")
                    return idx, result, was_truncated, None
                except Exception as exc:
                    failures.append(f"{backend}/{model}: {type(exc).__name__}: {str(exc)[:100]}")
                    await asyncio.sleep(MIN_INTERVAL_SEC)
            return idx, {"experiments": []}, False, "; ".join(failures[-4:])

    async def run_all():
        sem = asyncio.Semaphore(max(1, int(G4F_CONCURRENCY)))
        return await asyncio.gather(*(one_fragment(i, piece, pages, sem)
                                     for i, (piece, pages) in enumerate(fragments, 1)))

    results = _run_coroutine(run_all())
    experiments, errors, truncated = [], [], False
    for idx, result, was_truncated, error in sorted(results, key=lambda x: x[0]):
        if error:
            errors.append(f"фрагмент {idx}: {error}")
            print(f"  ⚠ фрагмент {idx}: пропущен после перебора провайдеров")
            continue
        truncated = truncated or was_truncated
        experiments.extend(result.get("experiments") or [])
    return experiments, truncated, errors

def model_issues():
    """Краткий аудит каскада в issues формы: режим и почему (provenance: кем извлекались данные)."""
    issues = [f"model_mode: {MODEL_MODE}", f"active_model: {MODEL or 'pollinations (fallback)'}"]
    failed = [f"{r['model']} — {r['vision_why'] or r['alive_why']}" for r in MODEL_AUDIT
              if r.get("alive") and r.get("vision") is False]
    if MODEL_MODE != "vision" and ENABLE_VISION:
        issues.append("vision-контур не работает: мультимодальные модели недоступны; "
                      "значения с картинок помечены null")
    for line in failed[:5]:
        issues.append(f"vision-ping: {line}")
    return issues

# --- детерминированный фолбэк по регулярным правилам (только при ALLOW_RULES_FALLBACK = True) ---
NUM_RE = re.compile(r"(?<![A-Za-z0-9.])([-+]?\d(?:[\d\s,]{0,12}\d)?(?:\.\d+)?)(?:\s*(?:±|\+/-)\s*([\d.]+))?\s*([%‰A-Za-zµμ°/^][A-Za-z0-9µμ°/^·._-]{0,15})?")
STAT_WORDS = re.compile(r"(?i)\b(n\s*=\s*\d+|p\s*[<=]\s*[\d.]+|R2|r\s*=\s*-?[\d.]+|SD|SE|CI|mean|median|yield|rate|concentration|dose|temperature)\b")

def rule_extract(doc):
    """Черновик без сети: строки с числами и статистическими словами. Низкая точность, всё — кандидаты."""
    experiments, current = [], None
    for n, line in enumerate(PAGE_RE.sub("", doc["text"]).splitlines(), 1):
        if re.match(r"(?i)^\s*(?:\d+(\.\d+)*\s+[A-Z][^.]{5,80}|(?:results|materials and methods|experiment\s*\d+|table\s*\d+|figure\s*\d+))", line) and len(line) < 120:
            current = {"experiment_id": f"exp-{len(experiments) + 1}", "profile": "experiment", "title": line.strip()[:160],
                       "research_question": None, "hypothesis": None, "design": None, "sample_size": None, "population": None,
                       "methods": None, "intervention": None, "control": None, "conclusion": None, "results": []}
            experiments.append(current)
        if current is None or not STAT_WORDS.search(line):
            continue
        for m in NUM_RE.finditer(line):
            current["results"].append({
                "page": 1, "modality": "table" if line.count("|") >= 2 or line.count("\t") >= 2 else "text",
                "locator": f"строка {n}", "field": "не определено", "value": m.group(1).replace(" ", ""),
                "unit": m.group(3), "value_type": "number", "confidence": 0.3, "status": "candidate",
                "evidence_quote": line.strip()[:400], "reviewer_note": ""})
            if len(current["results"]) >= MAX_RECORDS_PER_EXPERIMENT:
                break
        if len(experiments) >= MAX_EXPERIMENTS:
            break
    return experiments

print("модуль извлечения готов: фрагмент → JSON experiments → records")

## 7. Проверка ответа модели и provenance

До создания формы каждый кандидат проходит проверки:
- **value/unit** — значение дословное; `null` допускается, но помечается для эксперта;
- **evidence_quote** — ищется в тексте статьи (с нормализацией пробелов); если не найдено, запись отбрасывается как
  выдуманная и попадает в предупреждения;
- **page** — определяется по позиции цитаты и сверяется со страницей от модели; расхождение показывается;
- **modality** — `table`, если фрагмент содержит разделители колонок; `figure` без цитаты невозможен;
- **locator** должен называть место (`Table 2 / row ... / column ...`, `Figure 1 panel b`, `p.3 para 2`);
- **confidence** — нормируется в `[0, 1]` и снижается при короткой цитате, пустом значении или сомнительной modality.

Проверки не выносят решение за эксперта: они только отсекают недоказуемые значения и показывают проблемы в форме.

In [ ]:
import secrets as _secrets

def _squash(text):
    return re.sub(r"\s+", " ", text or "").strip()

def _find_offset(doc, anchor):
    """Позиция дословной цитаты в тексте статьи или -1 (пробелы нормализуются)."""
    target = re.sub(r"\s+", "", _squash(anchor))
    if len(target) < 8:
        return -1
    positions = [i for i, ch in enumerate(doc["text"]) if not ch.isspace()]
    hit = "".join(doc["text"].split()).find(target)
    if hit < 0 or hit >= len(positions) or hit + len(target) - 1 >= len(positions):
        return -1
    return positions[hit]

def _paragraph_at(text, start):
    head = text.rfind("\n\n", 0, start)
    start = 0 if head < 0 else head + 2
    tail = text.find("\n\n", start)
    end = len(text) if tail < 0 else tail
    return text[start:end].strip()

def _norm_page(value, pages):
    try:
        page = int(value)
    except (TypeError, ValueError):
        page = 1
    return min(max(page, 1), max(pages, 1))

def _norm_value(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return str(value).lower()
    if isinstance(value, float):
        return f"{value:.10g}"
    return str(value).strip()[:200] or None

def _norm_record(r, doc, problems):
    r = dict(r or {})
    for alias, canonical in FIELD_ALIASES.items():
        if alias in r and canonical not in r:
            r[canonical] = r.pop(alias)
    quote = str(r.get("evidence_quote") or "").strip()
    value, unit = _norm_value(r.get("value")), _norm_value(r.get("unit"))
    field_name = re.sub(r"\s+", " ", str(r.get("field") or "")).strip()[:120] or "не определено"
    try:
        confidence = max(0.0, min(1.0, float(r.get("confidence") if r.get("confidence") is not None else 0.3)))
    except (TypeError, ValueError):
        confidence = 0.3
    modality = str(r.get("modality") or "text").lower()
    offset = _find_offset(doc, quote)
    page_model = _norm_page(r.get("page"), doc["pages"])
    if offset < 0:
        problems.append(f"«{field_name}»: цитата не найдена дословно в тексте — запись отклонена (нет свидетельства)")
        return None
    page_found = page_at(doc["offsets"], offset)
    notes = []
    if page_model != page_found:
        notes.append(f"страница от модели {page_model}, по цитате {page_found} — принята {page_found}")
    if modality not in MODALITIES:
        modality = "text"
    para = _paragraph_at(doc["text"], offset)
    table_like = para.count("|") >= 2 or para.count("\t") >= 2
    if modality == "table":
        # Цитата из обычного абзаца не может подтверждать табличную ячейку.
        if not table_like:
            modality = "text"
            notes.append("modality=table, но в найденном фрагменте нет табличных разделителей — снижено до text")
            confidence = min(confidence, 0.5)
    elif modality == "text" and table_like and quote.count("|") >= 2:
        modality = "table"
        notes.append("цитата содержит табличные разделители — modality установлена как table")
    elif modality == "figure" and not re.search(r"(?i)\b(fig(?:ure)?\.?\s*\d+|panel\s+[a-z])", para):
        notes.append("modality=figure, но в цитате нет ссылки на рисунок")
        confidence = min(confidence, 0.5)
    para = _paragraph_at(doc["text"], offset)
    if len(_squash(quote)) < 15:
        notes.append("очень короткая цитата — сверьте locator")
        confidence = min(confidence, 0.45)
    if value is None:
        notes.append("значение не заявлено/неоднозначно: подтвердите null или впишите значение")
        confidence = min(confidence, 0.35)
    locator = re.sub(r"\s+", " ", str(r.get("locator") or "")).strip()[:200] or f"p.{page_found}"
    return {
        "record_id": str(r.get("record_id") or "") or "rec-" + _secrets.token_hex(4),
        "document_id": doc["document_id"],
        "page": page_found,
        "modality": modality,
        "locator": locator,
        "field": field_name,
        "value": value,
        "unit": unit,
        "value_type": r.get("value_type") if r.get("value_type") in VALUE_TYPES else "number",
        "confidence": round(confidence, 2),
        "status": "candidate",
        "evidence_quote": quote[:1200],
        "reviewer_note": "; ".join(notes)[:600],
    }

def _quotes_for(exp, doc):
    """Цитаты-подтверждения для полей эксперимента (design, sample_size, methods, ...)."""
    out, seen = [], set()
    for key in ("sample_size", "design", "intervention", "control", "methods", "conclusion", "research_question"):
        value = str(exp.get(key) or "").strip()
        if len(value) < 6:
            continue
        offset = _find_offset(doc, value)
        if offset < 0:
            continue
        snippet = _paragraph_at(doc["text"], offset)[:400]
        if snippet and snippet not in seen:
            seen.add(snippet)
            out.append({"field": key, "page": page_at(doc["offsets"], offset), "evidence_quote": snippet})
    return out

def normalize_experiments(raw_experiments, doc, truncated, errors):
    issues, experiments, rejected = list(errors), [], 0
    if truncated:
        issues.insert(0, "ОТВЕТ МОДЕЛИ ОБОРВАН: часть значений могла не сохраниться — проверьте полноту по статье")
    seen = set()
    for idx, exp in enumerate(raw_experiments or [], 1):
        if not isinstance(exp, dict):
            continue
        results = []
        for raw in exp.get("results") or []:
            norm = _norm_record(raw, doc, issues)
            if norm is None:
                rejected += 1
                continue
            key = (norm["page"], _squash(norm["evidence_quote"]), norm["field"].lower(), str(norm["value"]), norm["unit"])
            if key in seen:
                continue
            seen.add(key)
            results.append(norm)
            if len(results) >= MAX_RECORDS_PER_EXPERIMENT:
                issues.append(f"эксперимент {idx}: обрезано до {MAX_RECORDS_PER_EXPERIMENT} записей")
                break

        def txt(key, limit=600):
            value = exp.get(key)
            return str(value).strip()[:limit] if value is not None and str(value).strip() else None

        experiments.append({
            "experiment_id": re.sub(r"[^A-Za-z0-9_.-]+", "-", str(exp.get("experiment_id") or f"exp-{idx}"))[:60] or f"exp-{idx}",
            "profile": "observational" if str(exp.get("profile") or "").lower().startswith("obs") else "experiment",
            "title": txt("title", 180) or f"эксперимент {idx}",
            "research_question": txt("research_question"),
            "hypothesis": txt("hypothesis"),
            "design": txt("design", 200),
            "sample_size": txt("sample_size", 120),
            "population": txt("population", 400),
            "methods": txt("methods", 800),
            "intervention": txt("intervention", 400),
            "control": txt("control", 400),
            "conclusion": txt("conclusion", 800),
            "results": results,
            "evidence_quotes": [q["evidence_quote"] for q in _quotes_for(exp, doc)[:3]],
        })
        if len(experiments) >= MAX_EXPERIMENTS:
            issues.append(f"в статье найдено больше {MAX_EXPERIMENTS} экспериментов — показаны первые")
            break
    issues.extend(f"эксперимент {i + 1}: нет записей с подтверждённой цитатой — впишите значения вручную"
                  for i, e in enumerate(experiments) if not e["results"])
    return experiments, issues, rejected

def build_package(doc, experiments, issues, method):
    total = len(doc["text"])
    if total > CFG["max_chars"] * 3:
        issues = list(issues) + [f"Статья большая ({total:,} символов): модель видит фрагменты по {CFG['max_chars']:,}; "
                                 "проверьте полноту по всем разделам с результатами"]
    return {
        "schema": "scientific-evidence-review.v1",
        "document": {"document_id": doc["document_id"], "filename": doc["filename"], "sha256": doc["sha256"],
                     "size_bytes": doc["size_bytes"], "chars": total, "pages": doc["pages"],
                     "license": CFG["license"]},
        "method": method,
        "model": CFG["model"],
        "run_id": CFG["run_id"],
        "experiments": experiments,
        "issues": issues,
        "stats": {"experiments": len(experiments), "records": sum(len(e["results"]) for e in experiments)},
    }

EXTRACTED = []
if DOCS:
    for doc in DOCS:
        print(f"\n=== {doc['filename']} ({doc['pages']} стр., {len(doc['text']):,} символов)")
        method, experiments, issues = None, [], []
        for attempt in (1, 2):
            try:
                raw_experiments, truncated, errors = llm_extract(doc)
                experiments, issues, rejected = normalize_experiments(raw_experiments, doc, truncated, errors)
                method = f"g4f:{CFG['model']}"
                issues.extend(model_issues())
                if rejected:
                    issues.insert(0, f"Отброшено записей без дословной цитаты: {rejected}")
                if sum(len(e['results']) for e in experiments):
                    break
                print("  записей с подтверждённой цитатой нет — повтор запроса")
            except Exception as exc:
                print(f"  попытка {attempt} не удалась: {str(exc)[:200]}")
        if not experiments:
            print("  ✗ провайдеры g4f не ответили: данные НЕ извлечены. Повторите запуск ячейки позже (бесплатный тариф бывает перегружен).")
            if not ALLOW_RULES_FALLBACK:
                print("  Для черновика на регулярных правилах (низкая точность) установите ALLOW_RULES_FALLBACK = True.")
                continue
            experiments, issues, _ = normalize_experiments(rule_extract(doc), doc, False, [])
            method = "rules-fallback"
            issues.insert(0, "МОДЕЛЬ НЕ ОТВЕТИЛА: данные извлечены регулярными правилами — точность низкая, проверяйте каждое поле")
        if not sum(len(e['results']) for e in experiments):
            issues.insert(0, "Записей с подтверждённой цитатой нет: форма создаётся с пустой выборкой — опишите в комментарии, что должно быть найдено")
        package = build_package(doc, experiments, issues, method)
        for iss in package["issues"][:8]:
            print("  •", iss)
        print(f"  экспериментов: {package['stats']['experiments']} | записей: {package['stats']['records']}")
        EXTRACTED.append(package)

## 8. Автономная офлайн HTML-форма

Форма — один HTML-файл без внешних ресурсов и сервера: JSON извлечения встроен в `<script type="application/json">`.
По каждому значению эксперт выбирает **«верно / исправить / нет в источнике»**, а для `value/unit/field` дополнительно
можно подтвердить **`null`**. Итоговый статус записи: `approved` — если значение «верно» (или исправлено) и ни одно поле
не помечено «нет в источнике»; иначе `rejected`. Есть оценка 0–10, комментарий, заметка по каждому значению, кнопки
«Скачать JSON» и «Опубликовать» — обе берут **только одобренные** записи.

Отправка из браузера: один запрос `POST /api/datasets/{repo}/commit/{branch}?create_pr=1` в формате NDJSON
(`header` + `file` с base64) с `Authorization: Bearer <токен>`; ветка берётся из `refs`, отдельный `preupload` не нужен.
Токен живёт только в памяти страницы (не пишется в localStorage). При 404 форма объясняет, что датасет не найден или
токену не выдан доступ; при 401/403 — что нужен токен с правом записи. Если браузер заблокирует запрос, скачайте
JSON и выполните ячейку 10.

Если Run all выводит «Предупреждение о доверенном выводе», откройте `expert_review_*.html` из панели 📁 Colab или
выберите «Доверять» для ячейки — содержимое формы при этом не меняется.

In [ ]:
FORM_CSS = """
:root { --bg:#f7f7f5; --card:#fff; --ink:#1c1c1e; --muted:#6b6b70; --line:#e3e1dc; --ok:#1a7f4b; --warn:#b4690e; --bad:#b3202a; --accent:#1f4e46; }
* { box-sizing:border-box; }
body { margin:0; background:var(--bg); color:var(--ink); font:15px/1.55 -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif; }
header { background:var(--accent); color:#fff; padding:22px 28px; }
header h1 { margin:0 0 4px; font-size:21px; font-weight:600; } header p { margin:0; opacity:.85; font-size:13px; }
main { max-width:1120px; margin:0 auto; padding:22px 18px 80px; }
.card { background:var(--card); border:1px solid var(--line); border-radius:10px; padding:18px 20px; margin-bottom:18px; }
.card h2 { margin:0 0 12px; font-size:16px; font-weight:600; } .card h2 .tag { font-size:12px; font-weight:500; color:var(--muted); }
.grid { display:grid; grid-template-columns:200px 1fr; gap:6px 14px; font-size:14px; } .grid dt { color:var(--muted); } .grid dd { margin:0; word-break:break-word; }
.muted { color:var(--muted); font-size:13px; }
.banner { border-radius:10px; padding:12px 16px; margin-bottom:18px; font-size:14px; border:1px solid; }
.banner.bad { background:#fdf1f1; border-color:#f2c6c8; color:var(--bad); }
.badge { display:inline-block; padding:1px 8px; border-radius:20px; font-size:12px; border:1px solid var(--line); }
.badge.ok { color:var(--ok); border-color:#bfe3cd; background:#eef8f1; } .badge.bad { color:var(--bad); border-color:#f2c6c8; background:#fdf1f1; }
.exp { border:1px solid var(--line); border-radius:9px; padding:14px; margin-bottom:16px; }
.exp-head { display:flex; justify-content:space-between; gap:10px; align-items:baseline; margin-bottom:8px; flex-wrap:wrap; }
.exp-head b { font-size:16px; }
.kv { display:grid; grid-template-columns:190px 1fr; gap:2px 12px; font-size:13.5px; margin:6px 0 10px; } .kv dt { color:var(--muted); } .kv dd { margin:0; }
.rec { border:1px solid var(--line); border-radius:8px; padding:12px; margin-bottom:10px; background:#fdfdfc; }
.rec-head { display:flex; justify-content:space-between; gap:10px; align-items:baseline; margin-bottom:8px; flex-wrap:wrap; }
.rec-head b { font-size:14.5px; }
.field { border-top:1px dashed var(--line); padding:8px 0; }
.field .label { font-size:11.5px; color:var(--muted); text-transform:uppercase; letter-spacing:.03em; }
.field .value { margin:2px 0 6px; word-break:break-word; } .field .value.missing { color:var(--bad); font-style:italic; }
.choice { display:flex; flex-wrap:wrap; gap:6px; margin:2px 0 6px; }
.choice label { border:1px solid var(--line); border-radius:6px; padding:3px 10px; font-size:13px; cursor:pointer; background:#fafaf9; }
.choice input { margin-right:6px; } .choice label:has(input:checked) { background:#e9f1ef; border-color:#a9c8c0; }
input[type=text], input[type=password], textarea { width:100%; padding:7px 9px; border:1px solid var(--line); border-radius:6px; font:inherit; background:#fff; }
textarea { min-height:70px; resize:vertical; }
.fix { display:none; margin-top:6px; } .fix.show { display:block; }
.quote { border-left:3px solid var(--line); padding:4px 0 4px 12px; color:#3d3d42; background:#fbfbfa; white-space:pre-wrap; }
.warn-note { color:var(--warn); font-size:13px; }
.score-row { display:flex; align-items:center; gap:14px; flex-wrap:wrap; }
.score-row input[type=range] { flex:1; min-width:220px; } .score-row output { font-size:26px; font-weight:600; min-width:44px; text-align:center; }
.actions { display:flex; gap:10px; flex-wrap:wrap; align-items:center; margin-top:12px; }
button.primary { background:var(--accent); color:#fff; border:0; border-radius:7px; padding:10px 18px; font:inherit; cursor:pointer; }
button.secondary { background:#fff; border:1px solid var(--line); border-radius:7px; padding:9px 16px; font:inherit; cursor:pointer; }
button:disabled { opacity:.5; cursor:not-allowed; }
.status { margin-top:10px; font-size:14px; } .status.ok { color:var(--ok); } .status.bad { color:var(--bad); } .status.warn { color:var(--warn); }
.required-missing { outline:2px solid #f2c6c8; border-radius:6px; }
.consent { display:flex; gap:8px; align-items:flex-start; font-size:14px; margin:10px 0; } .consent input { margin-top:3px; }
code { background:#f1f1ef; padding:1px 5px; border-radius:4px; font-size:12px; }
"""

FORM_BODY = """<header><h1>Экспертная проверка извлечённых экспериментальных данных</h1>
<p>Результат сгенерирован моделью. Статус approved выставляет только эксперт; публикуются только одобренные записи.</p></header>
<main>
  <div class="banner" id="modeBanner" style="display:none"></div>
  <div class="card"><h2>Статья</h2><dl class="grid" id="docMeta"></dl><p class="muted" id="methodNote" style="margin:10px 0 0"></p></div>
  <div class="card" id="issuesCard" style="display:none"><h2>Автоматические предупреждения</h2><ul class="muted" id="issuesList" style="margin:0;padding-left:20px"></ul></div>
  <div class="card"><h2>Эксперименты и значения <span class="tag" id="recCount"></span></h2>
    <p class="muted">«верно» — совпадает со статьёй; «исправить» — укажите правильное значение; «нет в источнике» — лишнее; «null» — значение в статье не заявлено.</p>
    <div id="experiments"></div><div id="noRecords" style="display:none" class="muted">Записей нет. Опишите в комментарии, что должно было быть извлечено.</div></div>
  <div class="card"><h2>Оценка эксперта</h2>
    <div class="score-row"><input type="range" min="0" max="10" step="1" id="score" value="-1"><output id="scoreOut">—</output></div>
    <p class="muted">0 — не справился; 1–3 — в основном непригодно; 4–6 — часть данных, существенные ошибки; 7–8 — структура верна, отдельные ошибки; 9 — почти полное соответствие; 10 — полностью верно по проверенным полям.</p>
    <div class="field"><div class="label">Комментарий эксперта (обязателен при оценке ниже 10)</div><textarea id="comments" maxlength="2000" placeholder="Напр.: «Table 2: yield 7.2 t/ha — верно; значение 5.1 относится к другому варианту»"></textarea></div>
    <div class="field"><div class="label">Эксперт (имя или псевдоним, необязательно)</div><input type="text" id="expert" maxlength="80" placeholder="как подписать запись"></div></div>
  <div class="card"><h2>Публикация в Hugging Face</h2>
    <p class="muted">Публикуются только записи со статусом approved. Нужен токен с правом записи в датасет; он не сохраняется на диске.</p>
    <input type="password" id="hfToken" autocomplete="off" placeholder="hf_..."><div id="remoteHint" class="muted"></div>
    <div class="consent"><input type="checkbox" id="consent"><label for="consent">Я — эксперт и лично сверил(а) показанные значения со страницей, таблицей или рисунком статьи.</label></div>
    <div class="actions"><button class="primary" id="send">Опубликовать одобренные записи</button><button class="secondary" id="download">Скачать запись (JSON)</button><button class="secondary" id="reset">Сбросить пометки</button></div>
    <div class="status" id="status" role="status" aria-live="polite"></div></div>
  <p class="muted">Harness: <a href="https://github.com/top-papers/scientific-extraction-harness">scientific-extraction-harness</a> · Датасет: <a id="repoLink" target="_blank" rel="noopener"></a></p>
</main>"""

FORM_JS = """
(function () {
  'use strict';
  var DATA = JSON.parse(document.getElementById('review-data').textContent);
  var EXPS = DATA.experiments || [];
  var REPO = DATA.repo;
  var EXPFIELDS = [['title','Название'],['research_question','Исследовательский вопрос'],['hypothesis','Гипотеза'],['design','Дизайн'],['sample_size','Размер выборки'],['population','Популяция/объект'],['methods','Методы'],['intervention','Вмешательство'],['control','Контроль'],['conclusion','Вывод']];
  var RECFIELDS = [['field','field · переменная'],['value','value · значение'],['unit','unit · единица'],['value_type','value_type'],['page','page'],['modality','modality'],['locator','locator'],['confidence','confidence'],['evidence_quote','evidence_quote · цитата']];
  var JUDGEMENTS = [['correct','верно'],['fixed','исправить'],['absent','нет в источнике']];
  var NULLABLE = { value:true, unit:true, field:true };
  var state = {};
  function esc(s) { return String(s === null || s === undefined ? '' : s).replace(/[&<>"']/g, function (c) { return ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]); }); }
  function $(id) { return document.getElementById(id); }
  function show(v) { return (v === null || v === undefined || v === '') ? '<span class="missing">null / не указано</span>' : esc(v); }
  $('repoLink').textContent = REPO; $('repoLink').href = 'https://huggingface.co/datasets/' + REPO;
  var doc = DATA.document;
  $('docMeta').innerHTML = [['Файл', esc(doc.filename)],['document_id','<code>' + esc(doc.document_id) + '</code>'],['SHA-256','<code>' + esc(doc.sha256) + '</code>'],['Объём', doc.pages + ' стр. · ' + doc.chars.toLocaleString('ru-RU') + ' символов'],['Лицензия', esc(doc.license)],['Запись', esc(DATA.review_id) + ' · ' + esc(DATA.created_at)]].map(function (kv) { return '<dt>' + kv[0] + '</dt><dd>' + kv[1] + '</dd>'; }).join('');
  $('methodNote').textContent = 'Метод: ' + DATA.method + '. Полный текст статьи не публикуется.';
  if (String(DATA.method).indexOf('rules') === 0) { var b = $('modeBanner'); b.className = 'banner bad'; b.style.display = ''; b.textContent = '⚠ Модель не ответила: ниже черновик по регулярным правилам, проверяйте каждое значение.'; }
  if ((DATA.issues || []).length) { $('issuesCard').style.display = ''; $('issuesList').innerHTML = DATA.issues.map(function (x) { return '<li>' + esc(x) + '</li>'; }).join(''); }
  var totalRecs = EXPS.reduce(function (n, e) { return n + (e.results || []).length; }, 0);
  $('recCount').textContent = '— ' + EXPS.length + ' экспериментов · ' + totalRecs + ' записей';
  if (!totalRecs) $('noRecords').style.display = '';
  function fieldHtml(name, key, label, value, isQuote) {
    var choices = JUDGEMENTS.concat(NULLABLE[key] ? [['null','null']] : []);
    return '<div class="field" data-field="' + name + '"><div class="label">' + esc(label) + '</div>' +
      '<div class="value' + (isQuote ? ' quote' : '') + '">' + show(value) + '</div>' +
      '<div class="choice">' + choices.map(function (j) { return '<label><input type="radio" name="' + name + '" value="' + j[0] + '">' + j[1] + '</label>'; }).join('') + '</div>' +
      '<div class="fix"><input type="text" placeholder="Правильное значение (пусто = null)"></div></div>';
  }
  $('experiments').innerHTML = EXPS.map(function (e, i) {
    var results = e.results || [];
    var recs = results.length ? results.map(function (r, j) {
      var pk = i + '-' + j;
      return '<div class="rec" data-rec="' + pk + '"><div class="rec-head"><b>' + esc(r.field) + ' = ' + show(r.value) + ' ' + esc(r.unit || '') + '</b>' +
        '<span class="badge">стр. ' + esc(r.page) + ' · ' + esc(r.modality) + ' · ' + esc(r.locator) + '</span></div>' +
        RECFIELDS.map(function (f) { return fieldHtml(f[0] + ':' + pk, f[0], f[1], r[f[0]], f[0] === 'evidence_quote'); }).join('') +
        (r.reviewer_note ? '<div class="warn-note">⚠ ' + esc(r.reviewer_note) + '</div>' : '') +
        '<div class="field"><div class="label">Заметка проверяющего по значению</div><input type="text" class="recnote" data-rec="' + pk + '" placeholder="почему отклонили или исправили"></div></div>';
    }).join('') : '<p class="warn-note">Записей нет — опишите в комментарии, что должно было быть извлечено.</p>';
    return '<div class="exp" data-exp="' + i + '"><div class="exp-head"><b>' + (i + 1) + '. ' + esc(e.title) + '</b><span class="badge">' + esc(e.experiment_id) + ' · ' + esc(e.profile) + '</span></div>' +
      '<dl class="kv">' + EXPFIELDS.map(function (f) { return '<dt>' + esc(f[1]) + '</dt><dd>' + show(e[f[0]]) + '</dd>'; }).join('') + '</dl>' + recs + '</div>';
  }).join('');
  var boxes = document.getElementById('experiments');
  boxes.addEventListener('change', function (ev) {
    var t = ev.target; if (t.type !== 'radio') return;
    var parts = t.name.split(':'), key = parts[0], pk = parts[1];
    state[t.name] = { verdict: t.value, fix: '' };
    var fix = t.closest('.field').querySelector('.fix'), input = fix.querySelector('input');
    var needsFix = t.value === 'fixed';
    fix.classList.toggle('show', needsFix);
    if (needsFix) input.focus(); else { input.value = ''; }
  });
  boxes.addEventListener('input', function (ev) {
    if (ev.target.tagName !== 'INPUT' || ev.target.type !== 'text') return;
    var field = ev.target.closest('.field'); if (!field) return;
    if (state[field.dataset.field]) state[field.dataset.field].fix = ev.target.value;
  });
  var score = $('score'), scoreOut = $('scoreOut');
  function onScore() { var v = Number(score.value); scoreOut.textContent = v < 0 ? '—' : v; }
  score.addEventListener('input', onScore); onScore();
  function reviewOf(pk) {
    var out = {}, unchecked = 0;
    RECFIELDS.forEach(function (f) { var s = state[f[0] + ':' + pk]; if (s && s.verdict) out[f[0]] = s; else { out[f[0]] = null; unchecked++; } });
    return { fields: out, unchecked: unchecked };
  }
  function recordStatus(pk) {
    var rv = reviewOf(pk).fields, main = rv.value;
    var absent = Object.keys(rv).some(function (k) { return rv[k] && rv[k].verdict === 'absent'; });
    if (absent) return 'rejected';
    if (main && main.verdict === 'correct') return 'approved';
    if (main && main.verdict === 'fixed') return 'approved';
    if (main && main.verdict === 'null') return 'approved';
    return 'candidate';
  }
  function buildReview() {
    var sc = Number(score.value), comments = $('comments').value.trim();
    if (!(sc >= 0 && sc <= 10)) return { error: 'Выберите оценку от 0 до 10.', focus: score };
    if (sc < 10 && !comments) return { error: 'Добавьте комментарий: при оценке ниже 10 укажите ошибки явно.', focus: $('comments') };
    if (!$('consent').checked) return { error: 'Подтвердите, что проверка выполнена лично вами.', focus: $('consent').parentNode };
    var unchecked = 0, records = [];
    EXPS.forEach(function (e, i) { (e.results || []).forEach(function (r, j) {
      var pk = i + '-' + j, rv = reviewOf(pk), f = rv.fields;
      unchecked += rv.unchecked;
      var pick = function (k) { return (f[k] && (f[k].verdict === 'fixed' ? (f[k].fix || null) : (f[k].verdict === 'null' ? null : r[k]))); };
      var noteEl = document.querySelector('.recnote[data-rec="' + pk + '"]');
      records.push(Object.assign({}, r, {
        field: pick('field'), unit: pick('unit'), value: pick('value'),
        page: f.page && f.page.verdict === 'fixed' ? Number(f.page.fix) || r.page : r.page,
        locator: f.locator && f.locator.verdict === 'fixed' ? (f.locator.fix || r.locator) : r.locator,
        modality: f.modality && f.modality.verdict === 'fixed' ? (f.modality.fix || r.modality) : r.modality,
        evidence_quote: r.evidence_quote, experiment_id: e.experiment_id,
        status: recordStatus(pk),
        reviewer_note: [r.reviewer_note || '', noteEl ? noteEl.value.trim() : ''].filter(Boolean).join(' | ').slice(0, 600),
        field_reviews: f
      }));
    }); });
    if (unchecked && !confirm('Не отмечено полей: ' + unchecked + '. Продолжить?')) return { cancelled: true };
    var approved = records.filter(function (r) { return r.status === 'approved'; });
    return { review: { schema: 'scientific-evidence-review.v1', kind: 'expert_review', run_id: DATA.run_id, review_id: DATA.review_id, created_at: new Date().toISOString(), expert: $('expert').value.trim() || null, score: sc, comments: comments, expert_approved: true, document: DATA.document, method: DATA.method, model: DATA.model, repo: REPO, experiments: EXPS, records: records, approved_count: approved.length, issues: DATA.issues, unchecked_fields: unchecked } };
  }
  function setStatus(text, kind) { var s = $('status'); s.textContent = text; s.className = 'status ' + (kind || ''); }
  function markError(el) { Array.prototype.forEach.call(document.querySelectorAll('.required-missing'), function (x) { x.classList.remove('required-missing'); }); if (el && el.focus) { el.focus(); if (el.classList) el.classList.add('required-missing'); } }
  function download(obj) { var a = document.createElement('a'); a.href = URL.createObjectURL(new Blob([JSON.stringify(obj, null, 2)], { type: 'application/json' })); a.download = obj.review_id + '.json'; document.body.appendChild(a); a.click(); a.remove(); setTimeout(function () { URL.revokeObjectURL(a.href); }, 3000); }
  $('download').addEventListener('click', function () {
    var out = buildReview(); if (out.cancelled) return;
    if (out.review) { download(out.review); setStatus('Запись скачана: ' + out.review.review_id + '.json · одобрено ' + out.review.approved_count + ' записей', 'ok'); }
    else if (out.error) { setStatus(out.error, 'bad'); markError(out.focus); }
  });
  $('reset').addEventListener('click', function () {
    if (!confirm('Сбросить все пометки и комментарий?')) return;
    state = {};
    Array.prototype.forEach.call(document.querySelectorAll('#experiments input[type=radio]'), function (x) { x.checked = false; });
    Array.prototype.forEach.call(document.querySelectorAll('#experiments input[type=text]'), function (x) { x.value = ''; });
    Array.prototype.forEach.call(document.querySelectorAll('.fix'), function (x) { x.classList.remove('show'); });
    $('comments').value = ''; $('expert').value = ''; $('consent').checked = false; score.value = '-1'; onScore(); setStatus('Пометки сброшены.');
  });
  function b64utf8(text) { var bytes = new TextEncoder().encode(text), bin = ''; for (var i = 0; i < bytes.length; i++) bin += String.fromCharCode(bytes[i]); return btoa(bin); }
  $('send').addEventListener('click', async function () {
    var btn = $('send');
    if (!DATA.remote) { setStatus('Отправка из браузера отключена. Скачайте JSON и выполните ячейку 10.', 'warn'); return; }
    var out = buildReview(); if (out.cancelled) return;
    if (!out.review) { if (out.error) { setStatus(out.error, 'bad'); markError(out.focus); } return; }
    var review = out.review;
    if (!review.approved_count) { setStatus('Нет одобренных записей: публиковать нечего. Отметьте значения «верно».', 'bad'); return; }
    var token = $('hfToken').value.trim();
    if (!/^hf_[A-Za-z0-9_]+$/.test(token)) { setStatus(token ? 'Форма HF-токена неверна (ожидается hf_...).' : 'Введите HF-токен или скачайте запись.', 'warn'); return; }
    btn.disabled = true; setStatus('Отправка в Hugging Face…', 'warn');
    try {
      var approved = review.records.filter(function (r) { return r.status === 'approved'; });
      var artifact = { schema_version: 'reviewed-dataset.v1', document: review.document, expert_review: { review_id: review.review_id, expert: review.expert, score: review.score, comments: review.comments, created_at: review.created_at }, extraction_candidates: approved };
      var text = JSON.stringify(artifact, null, 2), content = b64utf8(text), path = 'validation/' + review.review_id + '.json';
      var auth = { 'Authorization': 'Bearer ' + token, 'Content-Type': 'application/json' };
      // Целевую ветку берём из refs: так работают и репозитории с другой веткой по умолчанию.
      var branch = 'main';
      try {
        var refs = await fetch('https://huggingface.co/api/datasets/' + REPO + '/refs', { headers: auth, cache: 'no-store' });
        if (refs.ok) { var refData = await refs.json(); if (refData && refData.branches && refData.branches.length) branch = refData.branches[0].name; }
      } catch (e) { branch = 'main'; }
      // NDJSON commit API (header/file) публикует напрямую; отдельный preupload не требуется.
      var body = JSON.stringify({ key: 'header', value: { summary: 'Add expert review ' + review.review_id } }) + '\\n' + JSON.stringify({ key: 'file', value: { path: path, content: content, encoding: 'base64' } }) + '\\n';
      var commit = await fetch('https://huggingface.co/api/datasets/' + REPO + '/commit/' + branch + '?create_pr=1', { method: 'POST', headers: { 'Authorization': 'Bearer ' + token, 'Content-Type': 'application/x-ndjson' }, body: body });
      var hfText = await commit.text();
      $('hfToken').value = '';
      if (!commit.ok) {
        var hint = commit.status === 404 ? ' Датасет ' + REPO + ' не найден или токену не выдан доступ: создайте датасет от имени аккаунта-владельца токена или укажите HF_REPO_ID и перезапустите ячейку 8.' : '';
        if (commit.status === 401 || commit.status === 403) hint = ' Проверьте права токена: нужен тип write (fine-grained: Write на этот датасет).';
        throw new Error('commit HTTP ' + commit.status + ': ' + hfText.slice(0, 300) + '.' + hint);
      }
      var data = {}; try { data = JSON.parse(hfText); } catch (e) { data = {}; }
      setStatus('Отправлено: ' + (data.pullRequestUrl || data.prUrl || data.commitUrl || (REPO + '/' + path)), 'ok');
    } catch (err) {
      var network = err && (err.name === 'TypeError' || /Failed to fetch|NetworkError|CORS/i.test(String(err.message)));
      setStatus('Не удалось отправить: ' + (err && err.message ? err.message : err) + (network ? ' Похоже, браузер заблокировал запрос (сеть/CORS).' : '') + ' Скачайте запись и выполните ячейку 10.', 'bad');
    }
    finally { btn.disabled = false; }
  });
  if (!DATA.remote) { $('remoteHint').textContent = 'Отправка из браузера отключена в ноутбуке (REMOTE_FORM_UPLOAD = False).'; $('send').disabled = true; }
})();
"""

def make_review_id(package):
    stem = re.sub(r"[^a-z0-9]+", "-", Path(package["document"]["filename"]).stem.lower()).strip("-")[:20] or "doc"
    return f"rv-{stem}-{package['document']['sha256'][:8]}"[:64]

def render_form(package, review_id):
    payload = json.dumps({
        "review_id": review_id, "run_id": package["run_id"], "created_at": CFG["created_at"], "repo": CFG["repo"],
        "method": package["method"], "model": package.get("model"), "remote": bool(REMOTE_FORM_UPLOAD),
        "document": package["document"], "experiments": package["experiments"], "issues": package["issues"],
    }, ensure_ascii=False).replace("</", "<\\/")
    return ("<!doctype html>\n<html lang=\"ru\">\n<head>\n<meta charset=\"utf-8\">\n"
            "<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n"
            "<title>Экспертная проверка — экспериментальные данные</title>\n<style>" + FORM_CSS + "</style>\n</head>\n<body>\n"
            + FORM_BODY +
            "\n<script id=\"review-data\" type=\"application/json\">" + payload + "</script>\n<script>" + FORM_JS + "</script>\n</body>\n</html>")

FORMS = []
FORM_PATH = Path("/content/expert_review.html") if os.path.isdir("/content") else Path("expert_review.html")
if not DOCS:
    print("Нет загруженных статей. Выполните ячейки раздела 3 (загрузка) и раздела 6 (извлечение), затем запустите эту ячейку снова.")
elif not EXTRACTED:
    print("Данные не извлечены (EXTRACTED пуст). Проверьте предупреждения g4f выше или поставьте ALLOW_RULES_FALLBACK = True для черновика.")
    print("Форма требует хотя бы один эксперимент с записями.")
else:
    for pkg in EXTRACTED:
        rid = make_review_id(pkg)
        out = Path(f"expert_review_{rid}.html")
        out.write_text(render_form(pkg, rid), encoding="utf-8")
        FORMS.append(out)
        print(f"✍️  {out} ({out.stat().st_size / 1024:.0f} КБ): экспериментов {pkg['stats']['experiments']}, "
              f"записей {pkg['stats']['records']} — оценка пока не выставлена")
    FORM_PATH = FORMS[0]
    print("Форма сохранена:", FORM_PATH)
    try:
        from IPython.display import HTML, display  # noqa: E402
        display(HTML(FORM_PATH.read_text(encoding="utf-8")))
    except Exception as exc:
        print("Не удалось показать форму inline:", type(exc).__name__, exc)
    print("Если форма не отрисовалась ниже, скачайте файл и откройте его в браузере:")
    try:
        from google.colab import files as colab_files  # type: ignore
        colab_files.download(str(FORM_PATH))
    except Exception:
        print("  путь к файлу:", FORM_PATH.resolve())

## 9. Открыть форму

Форма уже создана и показана в ячейке 8. Эта ячейка повторно выводит все формы (например, после повторного запуска
или для нескольких статей) — её можно пропустить и открыть скачанный HTML в любом браузере.
Для отправки прямо со страницы нужен интернет: форма ходит в HF API, пометки эксперта не покидают браузер.

In [ ]:
from IPython.display import HTML, display  # noqa: E402

for out in FORMS:
    print(out.name)
    display(HTML(out.read_text(encoding="utf-8")))


## 10. Публикация из ноутбука

Если отправка из формы сработала — этот блок ничего не делает для той записи.
Иначе скачайте JSON формы (`rv-*.json`), перетащите его в панель файлов Colab 📁 (корень `/content`) и выполните ячейку.

Перед публикацией запись проверяется: подтверждение эксперта, оценка 0–10, комментарий при оценке <10, у каждой
публикуемой записи `status: approved`, наличие полей `document_id/page/modality/locator/field/value/unit/confidence/evidence_quote`,
отсутствие токенов, e-mail и PDF-байтов. Токен берётся из секрета Colab и не печатается.

Публикуется **YAML в формате `reviewed-dataset.v1`** (как `export_reviewed_dataset` в пакете
`scientific-extraction-harness`) в `validation/<review_id>.yaml` через Commit API с `create_pr=1`.
Повторный запуск уже опубликованную запись не дублирует.


In [ ]:
import base64, hashlib, urllib.error

REQUIRED_RECORD_FIELDS = set(RESULT_FIELDS) - {"record_id", "reviewer_note"}
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
SECRET_RE = re.compile(r"\bhf_[A-Za-z0-9]{10,}|\bgh[pousr]_[A-Za-z0-9]{10,}|top_papers_token|HF_TOKEN\b")


def get_hf_token():
    token = _secret(HF_TOKEN_SECRET_NAME)
    if not token:
        print(f"⚠️ Токен не найден. Добавьте секрет {HF_TOKEN_SECRET_NAME} в 🔑 панели Colab (с доступом к блокноту) "
              "или задайте переменную окружения с тем же именем.")
        return None
    if not re.match(r"^hf_[A-Za-z0-9_]+$", token):
        print("⚠️ Значение секрета не похоже на HF-токен (ожидается hf_...).")
        return None
    return token


def _walk_strings(node, path="root"):
    if isinstance(node, dict):
        for k, v in node.items():
            if re.search(r"(^|_)(token|secret|api_key|apikey|password|passwd)($|_)", str(k).lower()):
                yield f"{path}.{k}", "secret-field"
            yield from _walk_strings(v, f"{path}.{k}")
    elif isinstance(node, list):
        for i, v in enumerate(node):
            yield from _walk_strings(v, f"{path}[{i}]")
    elif isinstance(node, str):
        yield path, node


def validate_review(rv):
    if rv.get("kind") != "expert_review":
        return "не запись экспертной оценки"
    if rv.get("expert_approved") is not True:
        return "нет явного подтверждения эксперта"
    score = rv.get("score")
    if isinstance(score, bool) or not isinstance(score, int) or not 0 <= score <= 10:
        return "оценка должна быть целым числом 0–10"
    if score < 10 and not str(rv.get("comments") or "").strip():
        return "при оценке ниже 10 нужен комментарий"
    if not re.match(r"^[a-z0-9][a-z0-9._-]{2,63}$", str(rv.get("review_id") or "")):
        return "некорректный review_id"
    records = rv.get("records")
    if not isinstance(records, list) or not records:
        return "в записи нет результатов извлечения"
    approved = [r for r in records if isinstance(r, dict) and r.get("status") == "approved"]
    if not approved:
        return "нет ни одной одобренной экспертом записи — публиковать нечего"
    for i, r in enumerate(approved, 1):
        missing = REQUIRED_RECORD_FIELDS - set(r)
        if missing:
            return f"запись {i}: нет полей {sorted(missing)}"
        if isinstance(r.get("page"), bool) or not isinstance(r.get("page"), int) or r["page"] < 1:
            return f"запись {i}: page должен быть целым >= 1"
        if r.get("modality") not in MODALITIES:
            return f"запись {i}: modality вне {MODALITIES}"
        try:
            if not 0 <= float(r.get("confidence")) <= 1:
                raise ValueError
        except (TypeError, ValueError):
            return f"запись {i}: confidence вне [0, 1]"
        if not str(r.get("evidence_quote") or "").strip():
            return f"запись {i}: нет evidence_quote"
    for path, value in _walk_strings(rv):
        if value == "secret-field":
            return f"в записи поле секрета: {path}"
        if SECRET_RE.search(value):
            return f"в записи значение, похожее на токен: {path}"
        if EMAIL_RE.search(value):
            return f"в записи найден e-mail (обезличьте): {path}"
        if "%PDF-" in value or "JVBERi0" in value:
            return f"в записи PDF-байты: {path}"
    return None


def _yaml_scalar(value):
    if value is None:
        return "null"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, int):
        return str(value)
    if isinstance(value, float):
        return repr(value)
    return json.dumps(str(value), ensure_ascii=False)


def _yaml_render(value, indent=0):
    pad, lines = " " * indent, []
    if isinstance(value, dict):
        if not value:
            return [pad + "{}"]
        for key in sorted(value):
            child = value[key]
            if isinstance(child, dict) and child:
                lines.append(f"{pad}{json.dumps(key, ensure_ascii=False)}:")
                lines.extend(_yaml_render(child, indent + 2))
            elif isinstance(child, list) and child:
                lines.append(f"{pad}{json.dumps(key, ensure_ascii=False)}:")
                lines.extend(_yaml_render(child, indent + 2))
            elif isinstance(child, dict):
                lines.append(f"{pad}{json.dumps(key, ensure_ascii=False)}: {{}}")
            elif isinstance(child, list):
                lines.append(f"{pad}{json.dumps(key, ensure_ascii=False)}: []")
            else:
                lines.append(f"{pad}{json.dumps(key, ensure_ascii=False)}: {_yaml_scalar(child)}")
        return lines
    if isinstance(value, list):
        if not value:
            return [pad + "[]"]
        for child in value:
            if isinstance(child, dict) and child:
                lines.append(pad + "-")
                lines.extend(_yaml_render(child, indent + 2))
            elif isinstance(child, list) and child:
                lines.append(pad + "-")
                lines.extend(_yaml_render(child, indent + 2))
            else:
                lines.append(pad + "- " + _yaml_scalar(child))
        return lines
    return [pad + _yaml_scalar(value)]


def export_reviewed_yaml(review):
    """Детерминированный YAML reviewed-dataset.v1 с теми же ключами, что в export_reviewed_dataset."""
    approved = [r for r in review["records"] if r.get("status") == "approved"]
    order = {e.get("experiment_id"): i for i, e in enumerate(review.get("experiments") or [])}
    approved.sort(key=lambda r: (order.get(r.get("experiment_id"), 999), r.get("record_id") or ""))
    candidates = [{k: r.get(k) for k in RESULT_FIELDS if k != "reviewer_note" or r.get(k)} for r in approved]
    artifact_hash = hashlib.sha256(json.dumps(approved, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    pseudonym = re.sub(r"[^A-Za-z0-9_-]+", "-", review.get("expert") or "expert")[:64] or "expert"
    package = {
        "article_hash": review["document"]["sha256"],
        "extraction_candidates": candidates,
        "expert_review": {"comment": review.get("comments") or None, "reviewer_pseudonym": pseudonym,
                          "score": review["score"]},
        "provenance": {"document_id": review["document"]["document_id"], "filename": review["document"]["filename"],
                       "pages": review["document"].get("pages"), "license": review["document"].get("license"),
                       "method": review.get("method"), "model": review.get("model"), "run_id": review["run_id"],
                       "created_at": review.get("created_at"), "repo": review.get("repo"),
                       "approved_records": len(approved), "artifact_sha256": artifact_hash},
        "schema_version": "reviewed-dataset.v1",
        "validation_event": {"candidate_id": review["document"]["document_id"],
                             "created_at": review.get("created_at") or dt.datetime.now(dt.timezone.utc).isoformat(),
                             "event_type": "expert_validation", "extraction_artifact_hash": artifact_hash,
                             "reviewer_pseudonym": pseudonym, "run_id": review["run_id"],
                             "validation_event_id": "validation-" + hashlib.sha256(
                                 f"{review['run_id']}|{review['review_id']}".encode()).hexdigest()[:24]},
    }
    return "\n".join(_yaml_render(package)) + "\n"


def hf_default_branch(repo, token):
    """Ветка по умолчанию из refs; при ошибке — main."""
    try:
        req = urllib.request.Request(f"https://huggingface.co/api/datasets/{repo}/refs",
                                     headers={"Authorization": f"Bearer {token}"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            branches = json.loads(resp.read().decode("utf-8") or "{}").get("branches") or []
        return branches[0]["name"] if branches else "main"
    except Exception:
        return "main"


def publish_review(review, token, repo=None):
    repo = repo or review.get("repo") or CFG["repo"]
    body = export_reviewed_yaml(review)
    path = f"validation/{review['review_id']}.yaml"
    branch = hf_default_branch(repo, token)
    payload = [{"key": "header", "value": {"summary": "Add expert review " + review["review_id"]}},
               {"key": "file", "value": {"path": path, "encoding": "base64",
                                         "content": base64.b64encode(body.encode("utf-8")).decode("ascii")}}]
    req = urllib.request.Request(
        f"https://huggingface.co/api/datasets/{repo}/commit/{branch}?create_pr=1",
        data="".join(json.dumps(item) + "\n" for item in payload).encode("utf-8"), method="POST",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/x-ndjson"})
    try:
        with urllib.request.urlopen(req, timeout=90) as resp:
            data = json.loads(resp.read().decode("utf-8") or "{}")
            link = data.get("pullRequestUrl") or data.get("prUrl") or data.get("commitUrl") or f"{repo}/{path}"
            return True, f"{resp.status} · {link}"
    except urllib.error.HTTPError as exc:
        hint = ""
        if exc.code == 404:
            hint = (f" Датасет {repo} не найден или токену не выдан доступ: создайте датасет от имени аккаунта-владельца"
                    " токена или поправьте HF_REPO_ID в ячейке 2.")
        elif exc.code in (401, 403):
            hint = " Проверьте права токена: нужен тип write (fine-grained: Write на этот датасет)."
        return False, f"HTTP {exc.code}: {exc.read().decode('utf-8', 'replace')[:300]}.{hint}"
    except Exception as exc:
        return False, f"{type(exc).__name__}: {str(exc)[:200]}"


SENT = {}
review_files = list(dict.fromkeys(sorted(Path("/content").glob("rv-*.json")) + sorted(Path(".").glob("rv-*.json"))))
if not review_files:
    print("Нет файлов rv-*.json. Скачайте запись из формы — или отправьте её прямо из формы.")
else:
    token = get_hf_token()
    for path in review_files:
        if path.name in SENT:
            print(f"✓ {path.name}: уже опубликовано")
            continue
        try:
            rv = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"✗ {path.name}: не JSON ({type(exc).__name__})")
            continue
        problem = validate_review(rv)
        if problem:
            print(f"✗ {path.name}: {problem}")
            continue
        if not token:
            print(f"… {path.name}: ожидает HF_TOKEN")
            continue
        ok, msg = publish_review(rv, token)
        print(("✓ " if ok else "✗ ") + f"{path.name}: {msg}")
        if ok:
            SENT[path.name] = msg


## 11. Скачать формы и манифест

Забирает HTML-формы, манифест запуска и zip из Colab. В манифест попадают только реквизиты и хэши — без текста статьи и без токена.


In [ ]:
import zipfile

manifest = [{"document_id": p["document"]["document_id"], "filename": p["document"]["filename"],
             "sha256": p["document"]["sha256"], "pages": p["document"]["pages"], "chars": p["document"]["chars"],
             "review_id": make_review_id(p), "method": p["method"], "model": p.get("model"),
             "experiments": p["stats"]["experiments"], "records": p["stats"]["records"], "run_id": p["run_id"]}
            for p in EXTRACTED]
Path("extraction_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

if FORMS:
    with zipfile.ZipFile("expert_forms.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for out in FORMS:
            z.write(out.name)
    try:
        from google.colab import files as colab_files  # type: ignore
        for out in FORMS:
            colab_files.download(str(out))
    except Exception:
        pass
    print("zip:", Path("expert_forms.zip").resolve())
print("manifest:", Path("extraction_manifest.json").resolve())
print("Готово: заполните форму, выставьте оценку 0–10 и опубликуйте только одобренные записи.")


## Ограничения и этика

- **Не научное заключение.** Ноутбук проверяет качество извлечения; решение о значении остаётся за экспертом.
- **g4f** использует публичные бесплатные провайдеры, доступность которых меняется. При сбое ноутбук повторяет запрос и
  сообщает об ошибке; HTML-форма с данными модели не создаётся без валидного JSON (черновик по правилам — только при
  `ALLOW_RULES_FALLBACK = True`).
- **Provenance обязателен:** значение без дословной цитаты отбрасывается до формы; страница определяется по цитате,
  расхождения показываются эксперту.
- **Таблицы и рисунки.** Текст таблиц извлекается как текст; значения, которые есть только на графике, модель прочитать
  не может — они должны быть `null` и заполнены экспертом вручную после просмотра рисунка.
- **PDF-сканы** без текстового слоя не распознаются (OCR нет). Ноутбук предупреждает, если текстовый слой пустой.
- **Публикуется только одобренная запись**: значения, provenance, оценка, комментарий и SHA-256. Полный текст статьи,
  PDF-байты, персональные данные и токены в датасет не попадают.
- **Токен** хранится только в секрете Colab или памяти HTML-страницы. Если он попал в чат, Git или логи — отозовите его
  в Hugging Face и создайте новый.
